In [1]:
import os, warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
print("LANGSMITH_TRACING :", os.getenv("LANGSMITH_TRACING", "not set"))
print("LANGSMITH_API_KEY :", "✅" if os.getenv("LANGSMITH_API_KEY") else "❌ missing")
print("GROQ_API_KEY :", "✅" if os.getenv("GROQ_API_KEY") else "❌ missing")
print("GEMINI_API_KEY :", "✅" if os.getenv("GEMINI_API_KEY") else "❌ missing")      

LANGSMITH_TRACING : true
LANGSMITH_API_KEY : ✅
GROQ_API_KEY : ✅
GEMINI_API_KEY : ✅


In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

# Plain llm.invoke() - no LCEL, no chain, no decorator
# Langsmith intercepts this call automatically via a three env variables we set 

response = llm.invoke("What is a langsmith run? Answer in 2 sentences only")
print(response.content)

A LangSmith run is an execution trace of a LangChain or LLM workflow captured by the LangSmith observability platform, which logs inputs, outputs, timings, and metadata for each step. It enables developers to monitor, debug, and analyze the performance and behavior of their language model pipelines.


In [4]:
import re
from langsmith import traceable


# -- Tool: Keyword search over the real document----------------------

@traceable(run_type="tool", name='doc_keyword_search')

def search_document(query= str, top_k: int = 3) -> list:
    """Searches llm_production_guide.txt by keyword overlap. Visible as a tool Run."""
    with open("data/llm_production_guide.txt", encoding="utf-8") as f:
        text = f.read()

    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 80]
    keywords = set(re.findall(r"\b\w{4,}\b", query.lower()))
    ranked = sorted(paragraphs, 
                    key=lambda p: sum(1 for kw in keywords if kw in p.lower()),
                    reverse=True)
    return ranked[:top_k]

In [5]:
# -- Chain: Orchestrates search -> LLM -> answer -------------------------------

@traceable(run_type = "chain", name = "doc_qa_pipeline")
def doc_qa(question: str) -> str:
    """Parent chain. LangSmith shows: doc_qa_pipeline -> dec_keyword_search + ChatGroq."""
    sections = search_document(question)
    context = "\n\n".join(sections)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer concisely:"
    return llm.invoke(prompt).content

In [6]:
answer = doc_qa("What are the main LLM security threats?")
print(f"Answer: {answer[:300]}...")

Answer: **Main LLM security threats (as highlighted by the OWASP Top 10 for LLM applications and production‑security guides)**  

1. **Prompt / Injection Attacks** – Malicious inputs that cause the model to execute unintended commands, reveal secrets, or produce harmful output.  
2. **Data Leakage & Privacy...


In [7]:
from langsmith import get_current_run_tree

@traceable(run_type="chain", name="support-query")
def support_qa(question: str, user_id:str, session_id:str) -> str:
    run = get_current_run_tree()
    if run:
        run.metadata.update({
            "user id" :     user_id,
            "session id" :  session_id,
            "feature" :     "customer_support",
            "env":          "production"
        })
        run.tags = ["production","support-bot","groq"]
    return llm.invoke(question).content

In [8]:
list_of_queries = [
    ("priya", "sess_001", "What is prompt injection and how we prevent it?"),
    ("aditi", "sess_002", "What are the best practices for the LLM output validation?"),
    ("sheetal", "sess_003", "How do we monitor LLM cost in production?"),
]

In [9]:
# Run 3 different users - each trace is tagged for filtering

for user, session, q in list_of_queries:
    answer = support_qa(q, user_id=user, session_id=session)
    print(f"[{user}{answer[:150]}....\n]")

[priya## Prompt Injection – A Quick Overview

**Prompt injection** is a class of attacks (or unintended behaviors) that arise when a language model (LLM) is....
]
[aditiBelow is a practical, step‑by‑step guide to validating the output of large language models (LLMs).  It is organized around **four validation layers** ....
]
[sheetal## Monitoring LLM Cost in Production – A Playbook

Below is a practical, end‑to‑end framework you can adopt (or cherry‑pick) to keep a tight leash on ....
]


# Excperiment no. 4 
Traced RAG over real file text file

Load data/llm_production_guide.txt, split into chunks, embed with google gemini(gemini-embedding-preview), build a FAISS index then wrap the whole RAG function in @traceable.

LangSmith shows the trace as:

production_guide_rag (run_type=chain)
        ChatGroq call (run_type=llm)
        input: full prompt WITH retrieved chunks
        output: answer
        tokens: input + output counts

The retrieved chunks and user_id are stored as meta data on the run - searchable in the dashboard.

In [10]:
import os 

from langsmith import get_current_run_tree #beacuse we want to trace the RAG
from langchain_community.document_loaders import TextLoader #beacuse the data is in the text format 
from langchain_text_splitters import RecursiveCharacterTextSplitter #to split the documents
from langchain_community.vectorstores import FAISS # to store data in vector database
from langchain_google_genai import GoogleGenerativeAIEmbeddings #to vertorize our data 



In [11]:
# Loading the data 

loader = TextLoader("data/llm_production_guide.txt", encoding="utf-8")
raw_docs = loader.load()
print(f"Loaded: {len(raw_docs[0].page_content):,} characters")

Loaded: 11,679 characters


In [12]:
# SPLIT The data
splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap=80)
chunks = splitter.split_documents(raw_docs)
print(f"chunks: {len(chunks)} (avg{sum(len(c.page_content) for c in chunks) //len(chunks)} chars each)")

chunks: 27 (avg430 chars each)


In [13]:
#Vectorize  
print("\nEmbedding chunks via Gemini API (gemini-embedding-2-Preview)...")
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=os.getenv("GEMINI_API_KEY"),

)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
print("✅ FAISS index ready")


Embedding chunks via Gemini API (gemini-embedding-2-Preview)...
✅ FAISS index ready


In [14]:
# --------------- Traced RAG Function - @traceable, no LCEL --------------------

@traceable(run_type="chain",user_id = str, name="production_guide_rag")

def rag(question: str, user_id:str = "anonymous")-> str:
    docs    = retriever.invoke(question)
    context ="\n\n".join(f"[chunk {i+1}] {d.page_content}" for i, d in enumerate(docs))
    prompt  = (
        f"Answer based ONLY on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer concisely:"
    )
    run = get_current_run_tree()
    if run:
        run.metadata.update({"user_id": user_id, "chunks_retrived": len(docs)})
    return llm.invoke(prompt).content

In [15]:
# Test with two questions ______________________________

for q, uid in[
    ("student_001" , "What are the main LLM risks in Production?"),
    ("student_002", "How would we evalute the LLM outputs for quality?"),
]:
    answer = rag(q, user_id=uid)
    print(f"\nQ:{q}")
    print(f"A: {answer[:250]}...")





Q:student_001
A: LLM-as‑Judge uses a powerful LLM (the judge) to score another LLM’s output against a reference. The highest‑priority risk is Prompt Injection (LLM01), which can be mitigated with semantic intent classifiers such as NeMo Guardrails. A secondary risk i...

Q:student_002
A: Insufficient information provided to answer the query....


In [16]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_community.utilities import GoogleSerperAPIWrapper

# Setup google serper

In [17]:
serper = GoogleSerperAPIWrapper()

tool

1. Retriver to (search the local docs)
2. Google search tool (to make search on google)  

In [18]:
@tool
def search_local_docs(query: str) -> str:
    """
    Search the internal LLM production guide.

    IMPORTANT:
    - Use only once per question.
    - After receiving the results, answer the user.
    - Do not call repeatedly.
    """
    docs = vectorstore.similarity_search(query, k=3)

    if not docs:
        return "No relevant documents found."

    response = "\n\n".join(
        f"[Chunk {i+1}]\n{doc.page_content[:700]}"
        for i, doc in enumerate(docs) 

    )

    #Prevent huge context windows

    return response[:2500] 

In [19]:
@tool
def google_search(query: str) -> str:
    """
    Search the web for recent information.

    IMPORTANT:
    - Use only once per question
    - After receiving the results, answer the user.
    - Do not search again unless absolutely required.

    """

    try: 
        result = serper.run(query)

        if not result:
            return "No search results found."

        return str(result)[:2500]
    except Exception as e:
        return f"Search failed: {str(e)}"        

REACT AGENT

In [20]:
agent = create_agent(
    model = llm,
    tools = [search_local_docs,google_search],
    system_prompt="""

    You are a research assistant.
You have two tools:

1. search_local_docs
    - Use RAG, security, evaluation, monitoring,
    prompt engineering, guardrails, deployment.

2. google_search
    - Use for current events, news,
    regulations, recent AI developments.

Rules:

1.Call a tool only if needed
2.Never call the same tool more than once.
3.Maximum TWO total tool calls.
4.After receiving tool results, provide final answer.
5.Do NOT continue searching if enough information exists.
6.Do NOT loop.
7.If one tool gives sufficient information,
    answer immidiately.
"""
    
)

# Runner Agent


In [21]:
def run_agent(question: str):

    result = agent.invoke(
        {
            "messages":[
                HumanMessage(content=question)
            ]
        },
        config = {
            "recursion_limit": 10  
        }
    )

    tools_used = []

    for msg in result['messages']:
        if isinstance(msg, ToolMessage):
            tools_used.append(msg.name)

    final_answer = ""

    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage):
            final_answer = msg.content
            break

    return final_answer, list(dict.fromkeys(tools_used))

In [22]:
queries=[
    ("What are LLM prompt injection attacks and how do we defend against them?",
     "search_local_docs"
     ),
     ("What are the latest AI regulations passed in 2025",
      "google_search"
      ),
      ("How does RAG work and what are the latest open-source RAG frameworks in 2025?",
        "both"          
      )
]

RUN, EVALUATE AND OBSERVE

In [23]:
for question, expected in queries:

    print("\n" + "=" * 80)
    print("QUESTION:")
    print(question)

    print("\n EXPECTED:")
    print(expected)

    answer, tools = run_agent(question)

    print("\nTOOLS USED:")
    print(tools)

    print("\nANSWER:")
    print(answer[:500])

print("\n ✅ Completed Successfully")


QUESTION:
What are LLM prompt injection attacks and how do we defend against them?

 EXPECTED:
search_local_docs

TOOLS USED:
['search_local_docs']

ANSWER:
**LLM prompt‑injection attacks – what they are**

A *prompt‑injection* attack is a technique in which an adversary embeds malicious instructions into the text that the language model receives, with the goal of overriding or subverting the model’s intended behavior.  
Two main flavors are commonly discussed:

| Type | How it works | Typical example |
|------|--------------|-----------------|
| **Direct injection** | The attacker places a hostile instruction **directly in the user‑message** that t

QUESTION:
What are the latest AI regulations passed in 2025

 EXPECTED:
google_search

TOOLS USED:
['google_search']

ANSWER:
**Key AI‑Regulatory Developments that were enacted or formally passed in 2025**

| Jurisdiction | Regulation (date of passage) | Scope & Highlights |
|---------------|------------------------------|----------------